# Distributed MPI + CUDA Stencil Lab

**Course:** Parallel Computing  
**GitHub username:** `nur5ek`  
**Execution:** Google Colab, T4 GPU, 4 MPI ranks

Run every code cell from top to bottom. Keep the real outputs in this notebook when uploading to GitHub.

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip -q install mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-0344a0d0-97da-f2de-bb45-be927a462707, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [2]:
GITHUB_USERNAME = "nur5ek"
assert GITHUB_USERNAME.strip()
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: nur5ek


In [3]:
%%writefile distributed_cuda_solver.py
import sys
import json
import math
import re
import hashlib
import subprocess
from pathlib import Path

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15
UP, DOWN, LEFT, RIGHT = range(4)
OPPOSITE = (DOWN, UP, RIGHT, LEFT)


def get_gpu_uuid():
    return subprocess.check_output(
        ['nvidia-smi', '--query-gpu=uuid', '--format=csv,noheader'],
        encoding='utf-8'
    ).strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    digest = hashlib.sha256(f'{github_user.strip()}::{gpu_uuid}'.encode('utf-8')).hexdigest()
    return int(digest[:16], 16)


def init_distributed_context(github_user):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f'Expected exactly 4 MPI ranks, got {size}')
    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    seed = derive_student_seed(github_user, gpu_uuid)
    cart = comm.Create_cart([2, 2], periods=[False, False], reorder=False)
    row_rank, col_rank = cart.Get_coords(cart.Get_rank())
    nbr_up, nbr_down = cart.Shift(0, 1)
    nbr_left, nbr_right = cart.Shift(1, 1)
    return {
        'cart': cart, 'rank': cart.Get_rank(), 'size': size,
        'coords': (row_rank, col_rank),
        'neighbors': (nbr_up, nbr_down, nbr_left, nbr_right),
        'gpu_uuid': gpu_uuid, 'seed': seed, 'github_username': github_user,
        'ny': GLOBAL_NY // 2, 'nx': GLOBAL_NX // 2,
    }


def initialize_subdomains(ctx):
    ny, nx = ctx['ny'], ctx['nx']
    row_rank, col_rank = ctx['coords']
    gy = row_rank * ny + np.arange(ny + 2, dtype=np.float64)
    gx = col_rank * nx + np.arange(nx + 2, dtype=np.float64)
    y = gy[:, None] / (GLOBAL_NY + 1)
    x = gx[None, :] / (GLOBAL_NX + 1)
    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * x) * np.cos(2.0 * np.pi * y)
    seed = ctx['seed']
    cx1 = 0.25 + ((seed >> 0) & 255) / 255.0 * 0.15
    cy1 = 0.25 + ((seed >> 8) & 255) / 255.0 * 0.15
    cx2 = 0.60 + ((seed >> 16) & 255) / 255.0 * 0.15
    cy2 = 0.60 + ((seed >> 24) & 255) / 255.0 * 0.15
    amp1 = 0.8 + ((seed >> 32) & 255) / 255.0 * 0.5
    amp2 = 0.5 + ((seed >> 40) & 255) / 255.0 * 0.4
    u = (amp1 * np.exp(-((x-cx1)**2 + (y-cy1)**2) / 0.018)
         + amp2 * np.exp(-((x-cx2)**2 + (y-cy2)**2) / 0.012))
    source = 0.0001 * np.exp(-((x-0.5)**2 + (y-0.5)**2) / 0.01)
    u = np.ascontiguousarray(u, dtype=np.float64)
    kappa = np.ascontiguousarray(kappa, dtype=np.float64)
    source = np.ascontiguousarray(np.broadcast_to(source, (ny+2, nx+2)))
    u[0, :] = 0.0
    u[-1, :] = 0.0
    u[:, 0] = 0.0
    u[:, -1] = 0.0
    return u, kappa, source


@cuda.jit
def pack_halos_kernel(u, out, direction, ny, nx):
    t = cuda.grid(1)
    if direction == UP and t < nx:
        out[t] = u[1, t + 1]
    elif direction == DOWN and t < nx:
        out[t] = u[ny, t + 1]
    elif direction == LEFT and t < ny:
        out[t] = u[t + 1, 1]
    elif direction == RIGHT and t < ny:
        out[t] = u[t + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, incoming, direction, ny, nx):
    t = cuda.grid(1)
    if direction == UP and t < nx:
        u[0, t + 1] = incoming[t]
    elif direction == DOWN and t < nx:
        u[ny + 1, t + 1] = incoming[t]
    elif direction == LEFT and t < ny:
        u[t + 1, 0] = incoming[t]
    elif direction == RIGHT and t < ny:
        u[t + 1, nx + 1] = incoming[t]


@cuda.jit
def tiled_stencil_kernel(u, v, kappa, source, dt, dx2, dy2,
                         r_start, r_end, c_start, c_end):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    r = r_start + cuda.blockIdx.y * TILE_DIM + ty
    c = c_start + cuda.blockIdx.x * TILE_DIM + tx
    smem = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)
    max_r = u.shape[0] - 1
    max_c = u.shape[1] - 1
    rr = min(max(r, 0), max_r)
    cc = min(max(c, 0), max_c)
    smem[ty + 1, tx + 1] = u[rr, cc]
    if tx == 0:
        smem[ty + 1, 0] = u[rr, max(cc - 1, 0)]
    if tx == TILE_DIM - 1:
        smem[ty + 1, TILE_DIM + 1] = u[rr, min(cc + 1, max_c)]
    if ty == 0:
        smem[0, tx + 1] = u[max(rr - 1, 0), cc]
    if ty == TILE_DIM - 1:
        smem[TILE_DIM + 1, tx + 1] = u[min(rr + 1, max_r), cc]
    cuda.syncthreads()
    if r_start <= r < r_end and c_start <= c < c_end:
        center = smem[ty + 1, tx + 1]
        north = smem[ty, tx + 1]
        south = smem[ty + 2, tx + 1]
        west = smem[ty + 1, tx]
        east = smem[ty + 1, tx + 2]
        kn = 0.5 * (kappa[r, c] + kappa[r - 1, c])
        ks = 0.5 * (kappa[r, c] + kappa[r + 1, c])
        kw = 0.5 * (kappa[r, c] + kappa[r, c - 1])
        ke = 0.5 * (kappa[r, c] + kappa[r, c + 1])
        diffusion = (ks * (south-center) - kn * (center-north)) / dy2
        diffusion += (ke * (east-center) - kw * (center-west)) / dx2
        v[r, c] = center + dt * (diffusion - LAMBDA_DAMP * center**3 + source[r, c])


def launch_box(u, v, k, s, dt, dx2, dy2, bounds, stream):
    r0, r1, c0, c1 = bounds
    if r1 <= r0 or c1 <= c0:
        return
    blocks = (math.ceil((c1-c0) / TILE_DIM), math.ceil((r1-r0) / TILE_DIM))
    tiled_stencil_kernel[blocks, (TILE_DIM, TILE_DIM), stream](
        u, v, k, s, dt, dx2, dy2, r0, r1, c0, c1)


def allocate_halos(ctx):
    halos = {}
    ny, nx = ctx['ny'], ctx['nx']
    for direction in (UP, DOWN, LEFT, RIGHT):
        length = nx if direction in (UP, DOWN) else ny
        halos[direction] = {
            'd_send': cuda.device_array(length, dtype=np.float64),
            'd_recv': cuda.device_array(length, dtype=np.float64),
            'h_send': cuda.pinned_array(length, dtype=np.float64),
            'h_recv': cuda.pinned_array(length, dtype=np.float64),
            'length': length,
        }
    return halos


def exchange_halos_async(ctx, u, halos, stream_boundary):
    neighbors = ctx['neighbors']
    for direction in range(4):
        if neighbors[direction] == MPI.PROC_NULL:
            continue
        h = halos[direction]
        blocks = math.ceil(h['length'] / 128)
        pack_halos_kernel[blocks, 128, stream_boundary](
            u, h['d_send'], direction, ctx['ny'], ctx['nx'])
        h['d_send'].copy_to_host(h['h_send'], stream=stream_boundary)
    stream_boundary.synchronize()
    requests = []
    cart = ctx['cart']
    for direction in range(4):
        neighbor = neighbors[direction]
        if neighbor != MPI.PROC_NULL:
            requests.append(cart.Irecv(halos[direction]['h_recv'],
                                       source=neighbor, tag=OPPOSITE[direction]))
    for direction in range(4):
        neighbor = neighbors[direction]
        if neighbor != MPI.PROC_NULL:
            requests.append(cart.Isend(halos[direction]['h_send'],
                                       dest=neighbor, tag=direction))
    return requests


def complete_halos(ctx, u, halos, requests, stream_boundary):
    MPI.Request.Waitall(requests)
    for direction in range(4):
        if ctx['neighbors'][direction] == MPI.PROC_NULL:
            continue
        h = halos[direction]
        h['d_recv'].copy_to_device(h['h_recv'], stream=stream_boundary)
        unpack_halos_kernel[math.ceil(h['length']/128), 128, stream_boundary](
            u, h['d_recv'], direction, ctx['ny'], ctx['nx'])


def step_overlapped(ctx, u, v, k, s, halos, dt, dx2, dy2,
                    stream_boundary, stream_interior, overlapped):
    ny, nx = ctx['ny'], ctx['nx']
    requests = exchange_halos_async(ctx, u, halos, stream_boundary)
    compute_stream = stream_interior if overlapped else stream_boundary
    if overlapped:
        launch_box(u, v, k, s, dt, dx2, dy2, (2, ny, 2, nx), compute_stream)
    complete_halos(ctx, u, halos, requests, stream_boundary)
    if not overlapped:
        launch_box(u, v, k, s, dt, dx2, dy2, (2, ny, 2, nx), compute_stream)
    launch_box(u, v, k, s, dt, dx2, dy2, (1, 2, 1, nx+1), stream_boundary)
    launch_box(u, v, k, s, dt, dx2, dy2, (ny, ny+1, 1, nx+1), stream_boundary)
    launch_box(u, v, k, s, dt, dx2, dy2, (2, ny, 1, 2), stream_boundary)
    launch_box(u, v, k, s, dt, dx2, dy2, (2, ny, nx, nx+1), stream_boundary)
    stream_boundary.synchronize()
    stream_interior.synchronize()
    return v, u


def global_diagnostics(ctx, u, previous, dx, dy):
    current = u.copy_to_host()[1:-1, 1:-1]
    old = previous.copy_to_host()[1:-1, 1:-1]
    energy_local = np.array([np.sum(current) * dx * dy], dtype=np.float64)
    energy_global = np.zeros(1, dtype=np.float64)
    ctx['cart'].Allreduce(energy_local, energy_global, op=MPI.SUM)
    residual_local = np.array([np.sum((current-old)**2)], dtype=np.float64)
    residual_global = np.zeros(1, dtype=np.float64)
    ctx['cart'].Allreduce(residual_local, residual_global, op=MPI.SUM)
    return float(energy_global[0]), float(np.sqrt(residual_global[0]))


def run_mode(ctx, initial, k, source, dt, dx, dy, overlapped):
    u = cuda.to_device(initial)
    v = cuda.to_device(np.zeros_like(initial))
    halos = allocate_halos(ctx)
    boundary_stream = cuda.stream()
    interior_stream = cuda.stream()
    cart = ctx['cart']
    trace = []
    init_energy = np.array([np.sum(initial[1:-1, 1:-1])*dx*dy], dtype=np.float64)
    global_energy = np.zeros(1, dtype=np.float64)
    cart.Allreduce(init_energy, global_energy, op=MPI.SUM)
    trace.append({'step': 0, 'energy': float(global_energy[0]), 'l2_residual': 0.0})
    cart.Barrier()
    begin = cuda.event(timing=True)
    end = cuda.event(timing=True)
    begin.record(boundary_stream)
    start = MPI.Wtime()
    for step in range(1, STEPS+1):
        u, v = step_overlapped(ctx, u, v, k, source, halos, dt, dx*dx, dy*dy,
                               boundary_stream, interior_stream, overlapped)
        if step % 100 == 0:
            energy, residual = global_diagnostics(ctx, u, v, dx, dy)
            trace.append({'step': step, 'energy': energy, 'l2_residual': residual})
    boundary_stream.synchronize()
    interior_stream.synchronize()
    end.record(boundary_stream)
    end.synchronize()
    wall_local_ms = (MPI.Wtime()-start)*1000.0
    wall_ms = cart.allreduce(wall_local_ms, op=MPI.MAX)
    event_local_ms = cuda.event_elapsed_time(begin, end)
    event_ms = cart.allreduce(event_local_ms, op=MPI.MAX)
    return u.copy_to_host(), trace, wall_ms, event_ms


def ptx_audit():
    signatures = tiled_stencil_kernel.signatures
    if not signatures:
        raise RuntimeError('Kernel did not compile')
    ptx = tiled_stencil_kernel.inspect_asm(signatures[0])
    counts = {
        'st_shared_f64': len(re.findall(r'\bst\.shared\.f64\b', ptx)),
        'ld_shared_f64': len(re.findall(r'\bld\.shared\.f64\b', ptx)),
        'bar_sync': len(re.findall(r'\bbar\.sync\b', ptx)),
        'st_shared_total': len(re.findall(r'\bst\.shared\.', ptx)),
        'ld_shared_total': len(re.findall(r'\bld\.shared\.', ptx)),
    }
    counts['passed'] = bool(counts['st_shared_total'] and
                            counts['ld_shared_total'] and counts['bar_sync'])
    return counts


def run_solver(github_user):
    ctx = init_distributed_context(github_user)
    rank = ctx['rank']
    initial, local_kappa, local_source = initialize_subdomains(ctx)
    kmax = ctx['cart'].allreduce(float(np.max(local_kappa)), op=MPI.MAX)
    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)
    dt_max = dx*dx / (4.0*kmax)
    dt = 0.35*dt_max
    k = cuda.to_device(local_kappa)
    source = cuda.to_device(local_source)
    if rank == 0:
        print('MPI ranks: 4; Cartesian grid: 2x2', flush=True)
        print(f'GitHub username: {github_user}', flush=True)
        print(f'GPU UUID: {ctx["gpu_uuid"]}', flush=True)
        print(f'INSTANCE_SEED: {ctx["seed"]}', flush=True)
        print(f'kappa_max={kmax:.12g} dx={dx:.12g} dt_max={dt_max:.12g} dt={dt:.12g}', flush=True)
    for label, overlap in (('serialized', False), ('overlapped', True)):
        if rank == 0:
            print(f'Running {label} mode: {STEPS} steps', flush=True)
        output, trace, wall_ms, cuda_ms = run_mode(ctx, initial, k, source, dt, dx, dy, overlap)
        if rank == 0:
            print(f'{label}_ms={wall_ms:.3f} cuda_event_ms={cuda_ms:.3f}', flush=True)
            for entry in trace:
                print(f'{label} step={entry["step"]} energy={entry["energy"]:.12g} '
                      f'l2_residual={entry["l2_residual"]:.12g}', flush=True)
        if not overlap:
            serialized = output
            serialized_trace = trace
            serialized_ms = wall_ms
            serialized_event_ms = cuda_ms
        else:
            overlapped_output = output
            overlapped_trace = trace
            overlapped_ms = wall_ms
            overlapped_event_ms = cuda_ms
    local_diff = float(np.max(np.abs(serialized[1:-1,1:-1] - overlapped_output[1:-1,1:-1])))
    parity = ctx['cart'].allreduce(local_diff, op=MPI.MAX)
    audit = ptx_audit()
    if rank == 0:
        print('PTX audit:', json.dumps(audit), flush=True)
        print(f'parity_max_diff={parity:.16g}', flush=True)
    manifest = {
        'github_username': github_user,
        'gpu_uuid': ctx['gpu_uuid'],
        'instance_seed': ctx['seed'],
        'mpi_ranks': ctx['size'],
        'cartesian_dims': [2, 2],
        'global_shape': [GLOBAL_NY, GLOBAL_NX],
        'local_padded_shape': [ctx['ny']+2, ctx['nx']+2],
        'steps': STEPS,
        'kappa_max': kmax,
        'dx': dx, 'dt': dt, 'dt_max': dt_max,
        'serialized_ms': serialized_ms,
        'overlapped_ms': overlapped_ms,
        'serialized_cuda_event_ms': serialized_event_ms,
        'overlapped_cuda_event_ms': overlapped_event_ms,
        'serialized_energy_trace': serialized_trace,
        'overlapped_energy_trace': overlapped_trace,
        'parity_max_diff': parity,
        'ptx_counts': audit,
        'ptx_audit_passed': audit['passed'],
        'verification_passed': audit['passed'] and parity < 1e-12,
    }
    if rank == 0:
        Path('submission_manifest.json').write_text(json.dumps(manifest, indent=2))
        print('Saved submission_manifest.json', flush=True)
        if not manifest['verification_passed']:
            raise RuntimeError('PTX or numerical parity verification failed')
    ctx['cart'].Barrier()


if __name__ == '__main__':
    if len(sys.argv) != 2 or not sys.argv[1].strip():
        raise SystemExit('Usage: python distributed_cuda_solver.py YOUR_GITHUB_USERNAME')
    run_solver(sys.argv[1].strip())


Writing distributed_cuda_solver.py


In [4]:
import subprocess
cmd = [
    'mpirun', '--allow-run-as-root', '--oversubscribe',
    '-np', '4', 'python3', 'distributed_cuda_solver.py', GITHUB_USERNAME
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print('STDERR:\n', proc.stderr)
    raise RuntimeError(f'MPI execution failed with exit code {proc.returncode}')

MPI ranks: 4; Cartesian grid: 2x2
GitHub username: nur5ek
GPU UUID: GPU-0344a0d0-97da-f2de-bb45-be927a462707
INSTANCE_SEED: 15315124151827154504
kappa_max=0.239999624971 dx=0.00194931773879 dt_max=3.95817248384e-06 dt=1.38536036934e-06
Running serialized mode: 400 steps
serialized_ms=10942.074 cuda_event_ms=10942.080
serialized step=0 energy=0.0754553079985 l2_residual=0
serialized step=100 energy=0.0754423504657 l2_residual=0.00314710358764
serialized step=200 energy=0.0754345060907 l2_residual=0.00310539134217
serialized step=300 energy=0.0754277306563 l2_residual=0.00307439378818
serialized step=400 energy=0.0754214646992 l2_residual=0.00304584693178
Running overlapped mode: 400 steps
overlapped_ms=1823.077 cuda_event_ms=1822.878
overlapped step=0 energy=0.0754553079985 l2_residual=0
overlapped step=100 energy=0.0754423504657 l2_residual=0.00314710358764
overlapped step=200 energy=0.0754345060907 l2_residual=0.00310539134217
overlapped step=300 energy=0.0754277306563 l2_residual=0.0

In [5]:
import json
from pathlib import Path
manifest_path = Path('submission_manifest.json')
assert manifest_path.exists(), 'Manifest was not generated'
m = json.loads(manifest_path.read_text())
print(json.dumps(m, indent=2))
assert m['github_username'] == GITHUB_USERNAME
assert m['mpi_ranks'] == 4
assert m['ptx_audit_passed'], 'PTX shared-memory verification failed'
assert m['parity_max_diff'] < 1e-12, 'Serialized vs overlap mismatch'
print('Basic verification checks passed')

{
  "github_username": "nur5ek",
  "gpu_uuid": "GPU-0344a0d0-97da-f2de-bb45-be927a462707",
  "instance_seed": 15315124151827154504,
  "mpi_ranks": 4,
  "cartesian_dims": [
    2,
    2
  ],
  "global_shape": [
    512,
    512
  ],
  "local_padded_shape": [
    258,
    258
  ],
  "steps": 400,
  "kappa_max": 0.23999962497115201,
  "dx": 0.001949317738791423,
  "dt": 1.3853603693425315e-06,
  "dt_max": 3.9581724838358046e-06,
  "serialized_ms": 10942.073529,
  "overlapped_ms": 1823.0765949999998,
  "serialized_cuda_event_ms": 10942.080078125,
  "overlapped_cuda_event_ms": 1822.877685546875,
  "serialized_energy_trace": [
    {
      "step": 0,
      "energy": 0.07545530799851399,
      "l2_residual": 0.0
    },
    {
      "step": 100,
      "energy": 0.0754423504656523,
      "l2_residual": 0.003147103587643152
    },
    {
      "step": 200,
      "energy": 0.07543450609072044,
      "l2_residual": 0.0031053913421686904
    },
    {
      "step": 300,
      "energy": 0.07542773065633

## Analytical & Forensic Engineering Responses

These answers use measurements from my completed Colab run on a Tesla T4 with 4 MPI ranks.

### Q1. Memory Coalescing & Strided Column Packing

In my 258 × 258 array, one float64 is 8 bytes. For a column the next value is 258 × 8 = 2064 bytes away, so its not continuous like a row. If I copied the whole range for one column, it would transfer 526328 bytes instead of only 2048 useful bytes. Thats 524280 extra bytes for one column, or 1048560 for both left and right. So I used the GPU pack kernel to put these values into a normal 1D buffer first. This makes the host transfer much smaller.


### Q2. Shared-Memory Tile Indexing & Bank Conflicts

My block is 16 × 16 and shared tile is 18 × 18. Thread (0,0) writes center `[1,1]` and halo `[0,1]`, `[1,0]`. It reads `[1,1]`, `[0,1]`, `[2,1]`, `[1,0]`, `[1,2]`. Thread (15,15) writes `[16,16]`, `[17,16]`, `[16,17]`. It reads `[16,16]`, `[15,16]`, `[17,16]`, `[16,15]`, `[16,17]`. T4 has 32 banks of 4 bytes, but float64 takes 8 bytes. So one value uses two bank words. Rows and columns can need multiple bank accesses, and 2-way conflicts can happen depending on lanes. Column stride is 18 × 8 = 144 bytes. PTX counts alone dont tell exact bank conflicts.


### Q3. PTX Assembly & Synchronization

My PTX result shows `st.shared.f64 = 5`, `ld.shared.f64 = 5`, `bar.sync = 1`. Audit passed. There is 256 threads in one block, so 8 warps. Without `cuda.syncthreads()`, one warp might read shared memory before another warp finishes writing halo values. Then result can be wrong. This barrier makes all threads wait before calculating.


### Q4. Four MPI Processes on One GPU

I got `serialized_ms = 10942.074` and `overlapped_ms = 1823.077`. Its around 6 times faster with overlap. CUDA event times are `10942.080` and `1822.878` ms. All 4 ranks use one T4 GPU, so they share GPU compute and copy resources. In overlapped version, interior work can run while MPI sends halo data. With 4 separate GPUs and InfiniBand it can behave different, because each rank have its own GPU. These time results is from my Colab run.


### Q5. MPI Deadlock & MPI.PROC_NULL

If all ranks use blocking `Send` before `Recv`, they might get stuck waiting for each other. Small messages can fit in MPI buffers, but big messages often wait until receiver posts `Recv`. Then nobody moves forward. Thats why I use `Irecv`, `Isend` and `Waitall`. For corner rank (0,0), there is no upper or left neighbor. MPI gives `MPI.PROC_NULL`, so those transfers are skipped and outer border stays zero.


### Q6. CFL Stability & Energy

My `kappa_max = 0.239999625` and `dx = 1/513 = 0.001949318`. The stability limit is `dt_max = dx² / (4 × kappa_max) = 3.95817248e-06`. My `dt = 1.38536037e-06`, which is 35% of the limit, so its below it.

My global energy from `Allreduce` was:

| Step | Energy |
|---:|---:|
| 0 | 0.0754553080 |
| 100 | 0.0754423505 |
| 200 | 0.0754345061 |
| 300 | 0.0754277307 |
| 400 | 0.0754214647 |

It keeps decreasing in both modes. Final difference was `0.0`, less than `1e-12`. So for my 400 steps simulation looks stable. I cant say it proves every future step will be stable.


In [6]:
from google.colab import files
files.download('mpi_cuda_stencil_lab.ipynb') if Path('mpi_cuda_stencil_lab.ipynb').exists() else print('Download the notebook using File > Download > Download .ipynb')
files.download('distributed_cuda_solver.py')
files.download('submission_manifest.json')

Download the notebook using File > Download > Download .ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>